# Momentum and rockets (Taylor 3.1-3.2)

*Class notebook, PHY 317.* Do each in-class problem on paper first, then run the matching section. Run the cells in order with **Shift-Enter**.

**Part I** is everything with no gravity in it: problems 3.5 and 3.10, plus the free-space rocket equation. **Part II** switches gravity on: problem 3.11.

Numbers for a Space Shuttle, roughly: empty mass $m_R = 130{,}000$ kg, takeoff mass $m_0 = 2{,}000{,}000$ kg (fuel is about 94% of the vehicle), exhaust speed $v_{\rm ex} = 3000$ m/s, burn rate $|\dot m| = 15{,}000$ kg/s.

In free space (Tsiolkovsky):
$$v_f = v_0 + v_{\rm ex}\ln\frac{m_0}{m_R}.$$
With gravity (problem 3.11), during the burn
$$v(t) = v_{\rm ex}\ln\frac{m_0}{m(t)} - g\,t, \qquad m(t) = m_0 - |\dot m|\,t .$$

## 1. Predict first (before you run anything)

Double-click the cell below, type, Shift-Enter.

- If the exhaust speed is doubled, does the final speed in free space double, more than double, or less than double?
- Roughly what fraction of the free-space burnout speed does gravity cost a Shuttle during its two-minute burn: about 1%, 10%, or 50%?

*(your prediction here)*

## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

m_rocket = 130e3      # kg, empty
m_0 = 2000e3          # kg, at takeoff
burn_rate = 15e3      # kg/s
t_burn = (m_0 - m_rocket) / burn_rate

---

# Part I. No gravity

Problems 3.5 and 3.10, then the free-space rocket equation. Nothing below feels $g$.

## 3. Equal masses, elastic collision (problem 3.5)

A projectile hits a stationary target of the *same* mass, elastically. Momentum gives $\vec v = \vec v_1{\,}' + \vec v_2{\,}'$ and energy gives $v^2 = v_1'^2 + v_2'^2$. Square the first, subtract the second, and $\vec v_1{\,}' \cdot \vec v_2{\,}' = 0$ -- the two always leave at right angles.

The dotted circle has $\vec v$ as its diameter. Whatever the scattering angle, the tip of $\vec v_1{\,}'$ lands on it: that circle *is* the energy condition.

In [ ]:
v0 = 1.0                                  # incoming speed, units do not matter here
angles = np.radians([20, 45, 70])         # lab angle of the scattered projectile

fig, ax = plt.subplots(1, 3, figsize=(11, 3.8))
for a, th in zip(ax, angles):
    v_in = np.array([v0, 0.0])
    v1 = v0 * np.cos(th) * np.array([np.cos(th), np.sin(th)])
    v2 = v_in - v1

    a.add_patch(plt.Circle((v0 / 2, 0), v0 / 2, fill=False,
                           linestyle=":", color="gray"))
    for vec, color, label in [(v_in, "gray", "v"), (v1, "C0", "v1'"),
                              (v2, "C1", "v2'")]:
        a.annotate("", xy=vec, xytext=(0, 0),
                   arrowprops=dict(arrowstyle="->", color=color, lw=2))
        a.text(1.06 * vec[0], 1.06 * vec[1], label, color=color)

    between = np.degrees(np.arccos(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2))))
    a.set_title(f"scattered at {np.degrees(th):.0f} deg\n"
                f"{between:.1f} deg apart", fontsize=11)
    a.set_aspect("equal")
    a.set_xlim(-0.25, 1.35)
    a.set_ylim(-0.75, 0.75)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("momentum:  v = v1' + v2'        energy:  v^2 = v1'^2 + v2'^2")
print("square the first, subtract the second:  2 v1'.v2' = 0, for every angle.")

## 4. When is the momentum largest? (problem 3.10)

In free space, starting from rest, $p = m v = m\,v_{\rm ex}\ln(m_0/m)$. You showed in class that this is largest when $m = m_0/e$. The burn runs **right to left**: the rocket starts at $m_0$ and ends at $m_R$.

In [ ]:
v_ex = 3000           # m/s
m = np.linspace(m_rocket, m_0, 400)
p = m * v_ex * np.log(m_0 / m)

plt.plot(m / 1e3, p)
plt.axvline(m_0 / np.e / 1e3, linestyle=":", color="gray")
plt.xlabel("mass still on the rocket (tonnes)")
plt.ylabel("momentum (kg m/s)")
plt.grid(alpha=0.3)
plt.show()

print(f"maximum at m = m_0/e = {m_0 / np.e / 1e3:.0f} tonnes, "
      f"p_max = m_0 v_ex/e = {m_0 * v_ex / np.e:.3g} kg m/s")
print(f"that is {(m_0 - m_0 / np.e) / burn_rate:.0f} s into the "
      f"{t_burn:.0f} s burn -- the speed keeps rising afterwards, but the mass falls faster.")

## 5. How much does the exhaust speed matter?

Final speed in free space as a function of $v_{\rm ex}$. The dotted line is the Shuttle's 3000 m/s.

In [ ]:
v_ex = np.linspace(100, 4000, 200)
v_final = v_ex * np.log(m_0 / m_rocket)

plt.plot(v_ex, v_final)
plt.axvline(3000, linestyle=":", color="gray")
plt.xlabel("exhaust speed v_ex (m/s)")
plt.ylabel("final speed (m/s)")
plt.grid(alpha=0.3)
plt.show()

print(f"burn time = {t_burn:.0f} s")
print(f"at v_ex = 3000 m/s, free-space burnout speed = {3000 * np.log(m_0 / m_rocket):.0f} m/s")

---

# Part II. Now switch gravity on

Problem 3.11. Everything below has $g$ in it, and that is the whole difference.

## 6. The gravity tax (problem 3.11)

Speed during the burn, with and without gravity. Move $g$ (0 = free space, 9.8 = Earth's surface, larger for a heavier planet) and the exhaust speed.

In [ ]:
@interact(g=(0.0, 20.0, 0.5), v_ex=(1000, 4500, 100))
def burn(g=9.8, v_ex=3000):
    t = np.linspace(0, t_burn, 400)
    m = m_0 - burn_rate * t
    v_free = v_ex * np.log(m_0 / m)
    v = v_free - g * t

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].plot(t, m / 1e3)
    ax[0].set_xlabel("t (seconds)")
    ax[0].set_ylabel("mass (tonnes)")
    ax[0].set_title("m(t)")
    ax[1].plot(t, v_free, "--", label="g = 0")
    ax[1].plot(t, v, label=f"g = {g}")
    ax[1].set_xlabel("t (seconds)")
    ax[1].set_ylabel("v (m/s)")
    ax[1].set_title("v(t) during the burn")
    ax[1].legend()
    for a in ax:
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"burnout speed: {v_free[-1]:.0f} m/s without gravity, {v[-1]:.0f} m/s with g = {g}"
          f"   (gravity tax {g * t_burn:.0f} m/s)")
    if burn_rate * v_ex < m_0 * g:
        print("Thrust is less than the initial weight: this rocket never leaves the pad (problem 3.11d).")

## 7. What you found

Double-click, type, Shift-Enter. How did it compare with your prediction?

*(what you found, compared with your prediction)*